# Senzai, Fernandez-Ruiz and Buzsaki (2019) from dandiset 000166

Paper: Senzai Y, Fernandez-Ruiz A, Buzsaki G (2019). Layer-specific physiological features and interlaminar interactions in the primary visual cortex of the mouse. Neuron 101, 500-513. doi:10.1016/j.neuron.2018.12.009

Dataset: DANDI:000166, version 0.250624.0434 (doi:10.48324/dandi.000166/0.250624.0434), streamed from the archive. The paper cites no version. The dandiset has two published versions, 0.220116.2037 and 0.250624.0434, and the cell below shows that they hold the same 19 assets (same ids, paths and sizes); the most recent published one is pinned, as the protocol asks when the paper cites none.

Claims: `claims/000166/10.1016-j.neuron.2018.12.009.yaml`, written from the version of record (PMC6367010) with its two supplements.

The dataset holds one session from each of 19 mice: spike-sorted units (spike times, a quality label and, in 18 files, cell type and synaptic effect labels), the raw 20 kHz signal and the 1250 Hz LFP of the 64 sites, brain state intervals (18 files), UP and DOWN intervals, and laser pulse times. It holds no unit waveforms, no depth of the sites, and no layer, landmark or cluster labels.

| Claim | Status here |
|---|---|
| fig4a-unit-counts | tested |
| fig4a-ccg-classification | tested |
| fig7-connection-counts | tested |
| fig7c-divergence-convergence | tested |
| figs8f-ie-dip-wake | tested |
| fig4de-optotagging | tested |
| figs7-strongest-3-6hz-coupling | tested |
| fig1g-landmark-distances-consistent | blocked (methods underspecified); every reading is run below |
| figs8c-lognormal-strength | blocked (methods underspecified, as the criterion says); described below |
| fig3ab-alpha-analogy | blocked (methods underspecified, as the criterion says) |
| fig1f-sink-b-deeper, fig1f-other-landmarks-same-depth, figs2d-visual-offset-rebound | blocked (data missing: visual stimulus times in 2 of the 13 mice) |
| fig3d-slow-oscillation-gamma-coupling, fig3d-3-6hz-cross-frequency, figs2c-3-6hz-theta-anticorrelation, fig3f-units-lock-3-6hz-trough | blocked (methods underspecified: their LFP sites are placed by the landmarks of fig1g) |
| 30 claims that depend, directly or through other claims, on fig1g-landmark-distances-consistent (the coherence clusters, gamma ICs, unit clusters and every layer claim) | blocked (methods underspecified, inherited) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy import stats
from concurrent.futures import ProcessPoolExecutor
import v1layers as v

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("dandiset", v.DANDISET_ID, "version", v.VERSION)
versions = requests.get(f"{v.API}/dandisets/{v.DANDISET_ID}/versions/", timeout=60).json()["results"]
ids = {}
for ver in versions:
    a = requests.get(f"{v.API}/dandisets/{v.DANDISET_ID}/versions/{ver['version']}/assets/?page_size=200",
                     timeout=60).json()["results"]
    ids[ver["version"]] = sorted((x["asset_id"], x["path"], x["size"]) for x in a)
    print(ver["version"], ver["status"], ver["asset_count"], "assets", round(ver["size"] / 1e9, 1), "GB")
published = [k for k in ids if k != "draft"]
print("published versions hold the same assets (ids, paths and sizes):",
      all(ids[k] == ids[published[0]] for k in published), "; draft the same:", ids["draft"] == ids[v.VERSION])

In [ ]:
assets = v.list_assets()
t0 = time.time()
with ProcessPoolExecutor(4) as ex:
    S = list(ex.map(v.process_session, assets))
print(f"{len(S)} files streamed and reduced in {time.time() - t0:.0f} s")

In [ ]:
# One row per unit and one per detected connection, then what was loaded.
rows = []
for si, r in enumerate(S):
    cls, cls_hp = v.classify_waveforms(r["wf"]), v.classify_waveforms(r["wf_hp"])
    for k in range(len(r["unit_ids"])):
        f, fh = r["wf"][k], r["wf_hp"][k]
        o = r["opto"]
        rows.append(dict(
            sess=si, subject=r["subject"], genotype=r["genotype"], unit=k, unit_id=r["unit_ids"][k],
            cell_type=r["cell_type"][k], synaptic_effect=r["synaptic_effect"][k], quality=r["quality"][k],
            max_channel=r["max_channel"][k], n_spikes=r["n_spikes"][k], wf_spikes=r["wf_spikes"][k],
            wf=cls[k], wf_hp=cls_hp[k], tp_ms=f["tp_ms"] if f else np.nan, tp_hp=fh["tp_ms"] if fh else np.nan,
            kurt=f["kurtosis"] if f else np.nan, site=f["site"] if f else -1, amp_uv=f["amp_uv"] if f else np.nan,
            ccgE=r["conn"]["exc"][k].any(), ccgI=r["conn"]["inh"][k].any(),
            ccgE_h=r["conn_hollow"]["exc"][k].any(), ccgI_h=r["conn_hollow"]["inh"][k].any(),
            opto_pooled=o["true_pooled"][k] if o else False, opto_any=o["true_any"][k] if o else False,
            opto_pooled_null=o["shifted_pooled"][k] if o else False, opto_any_null=o["shifted_any"][k] if o else False))
U = pd.DataFrame(rows)
crow = []
for si, r in enumerate(S):
    p = r["pairs"]
    u = U[U.sess == si].reset_index(drop=True)
    for j in range(len(p["pre"])):
        a, b = p["pre"][j], p["post"][j]
        crow.append(dict(sess=si, subject=r["subject"], pre=a, post=b, exc=p["exc"][j], inh=p["inh"][j],
                         pre_wf=u.wf[a], post_wf=u.wf[b], dsite=abs(u.site[a] - u.site[b]),
                         stp=p["stp"][j], dip_wake=p["dip_wake"][j], dip_nrem=p["dip_nrem"][j]))
C = pd.DataFrame(crow)

print("Files:", len(S), "subjects:", U.subject.nunique())
tab = []
for r in S:
    st = r["states"]
    dur = {lab: (st["stop"] - st["start"])[st["label"] == lab].sum() / 60 for lab in np.unique(st["label"])} if st else {}
    ud = r["updown"]
    tab.append(dict(subject=r["subject"], genotype=r["genotype"], session=r["session_id"], units=len(r["unit_ids"]),
                    good=int((r["quality"] == "good").sum()), spikes=int(r["n_spikes"].sum()),
                    hours=round(r["duration"] / 3600, 2), wake_min=round(dur.get("Awake", 0)), nrem_min=round(dur.get("Non-REM", 0)),
                    rem_min=round(dur.get("REM", 0)), ma_min=round(dur.get("MA", 0)),
                    UP=int((ud["label"] == "UP").sum()), DOWN=int((ud["label"] == "DOWN").sum()),
                    laser_pulses=len(r["laser"]["start"]) if r["laser"] else 0,
                    visual_pulses=len(r["visual"]["start"]) if r["visual"] else 0,
                    has_cell_type="cell_type" in r["columns"]))
T = pd.DataFrame(tab)
print(T.to_string(index=False))
print("\nTotals: units", T.units.sum(), "good", T.good.sum(), "spikes", T.spikes.sum())
print("Genotypes in the files:", T.genotype.value_counts().to_dict())
print("Genotypes in the paper (STAR Methods): 3 Ai35, 5 PV-Cre::Ai32, 9 VGAT-Cre::Ai32, 1 CaMKII-Cre::Ai35, 1 CaMKII-Cre::Ai32")
print("Files without a Sleep states table:", [r["subject"] for r in S if r["states"] is None])
print("Files without unit columns other than quality:", [r["subject"] for r in S if "cell_type" not in r["columns"]])
print("\nUnit quality:", U.quality.value_counts().to_dict())
print("Dataset cell_type labels:", U.cell_type.value_counts().to_dict())
print("Dataset synaptic_effect labels:", U.synaptic_effect.value_counts().to_dict())
print("Units labelled 'down state' by mouse:", U[U.cell_type == "down state"].subject.value_counts().to_dict())

print("\nSampling: raw rate", sorted({r["raw_rate"] for r in S}), "Hz; LFP rate", sorted({r["lfp_rate"] for r in S}),
      "Hz; raw sites", sorted({r["raw_shape"][1] for r in S}))
print("Electrode y (depth) all NaN in every file:", all(np.isnan(r["electrode_y"]).all() for r in S),
      "; rel_y range", sorted({(r["rel_y"].min(), r["rel_y"].max()) for r in S}))
print("Electrode locations:", sorted({x for r in S for x in r["electrode_location"]}),
      "; impedances:", sorted({float(x) for r in S for x in r["electrode_imp"]}))
print("Spike times sorted in every unit:", all(r["sorted_ok"].all() for r in S))
span_ok = [r["t_range"][0] >= 0 and r["t_range"][1] <= r["duration"] + 1 for r in S]
print("Spike times (s) within the raw recording in every file:", all(span_ok),
      "; latest spike / recording length:", round(min(r["t_range"][1] / r["duration"] for r in S), 3), "to",
      round(max(r["t_range"][1] / r["duration"] for r in S), 3))
rates = U.n_spikes / U.sess.map({i: r["duration"] for i, r in enumerate(S)})
print("Mean firing rates (Hz): median", round(rates.median(), 2), "range", round(rates.min(), 3), "to", round(rates.max(), 1))
same_site = (U.site >= 0)
print("Units whose `max_channel` equals the site of their largest mean waveform (column order of the raw data):",
      int((U.max_channel[same_site] == U.site[same_site]).sum()), "of", int(same_site.sum()))
heads = {}
for r in S:
    for h in r["train_head"]:
        heads.setdefault(h, []).append(r["subject"])
dups = {h: s for h, s in heads.items() if len(set(s)) > 1}
print("Spike trains whose first five spike times appear in more than one file:", len(dups))
print("Total recorded hours per file range:", T.hours.min(), "to", T.hours.max())
for r in S:
    if r["visual"] is not None:
        vs, vd = r["visual"]["start"], r["visual"]["stop"] - r["visual"]["start"]
        lz = r["laser"]["start"]
        near = np.min(np.abs(vs[:, None] - lz[None, :]), axis=1) < 0.005 if lz is not None and len(lz) else np.zeros(len(vs), bool)
        print(f"{r['subject']} 'Visual laser': {len(vs)} intervals, durations {np.percentile(vd, [0, 50, 100]).round(4)} s;"
              f" {near.sum()} start within 5 ms of a 'Laser diode' pulse; intervals of 0.09-0.11 s: {((vd > 0.09) & (vd < 0.11)).sum()},"
              f" of 0.39-0.41 s: {((vd > 0.39) & (vd < 0.41)).sum()}")
for r in S:
    if r["laser"] is not None and len(r["laser"]["start"]):
        d = r["laser"]["stop"] - r["laser"]["start"]
        if r["subject"] in ("YMV05", "YMV07"):
            print(f"{r['subject']} 'Laser diode': {len(d)} pulses, duration {np.percentile(d, [0, 50, 100]).round(4)} s,"
                  f" amplitude levels {r['opto']['levels']} with {r['opto']['n_pulses']} pulses")
print("Laser diode pulse counts per level by file:",
      {r["subject"]: r["opto"]["n_pulses"].tolist() for r in S if r["opto"] is not None})

In [ ]:
# Clock checks: spikes against the UP/DOWN table (true and shifted by 10 s), the UP/DOWN table against
# the state table, and spikes against the laser pulses (true and shifted by 0.5 s).
ck = []
for r in S:
    ur = r["updown_rates"]
    ok = (ur[("UP", 0.0)] > 0)
    true = np.nanmedian(ur[("DOWN", 0.0)][ok] / ur[("UP", 0.0)][ok])
    sh_ok = ur[("UP", 10.0)] > 0
    shifted = np.nanmedian(ur[("DOWN", 10.0)][sh_ok] / ur[("UP", 10.0)][sh_ok])
    ud, st = r["updown"], r["states"]
    if st is not None:
        nm = st["label"] == "Non-REM"
        mid = (ud["start"] + ud["stop"]) / 2
        inside = np.zeros(len(mid), bool)
        for a, b in zip(st["start"][nm], st["stop"][nm]):
            inside |= (mid >= a) & (mid < b)
        frac = inside.mean()
    else:
        frac = np.nan
    L = r["landmark"]
    lfp_sd = L["lfp"].std(axis=0).max()            # largest SD over time of the transition-averaged LFP, over sites
    lfp_sd_shift = L["lfp_shifted"].std(axis=0).max()
    ck.append(dict(subject=r["subject"], down_up_ratio=round(true, 3), down_up_ratio_shifted=round(shifted, 3),
                   updown_in_nrem=round(frac, 3), lfp_transition_uv=round(lfp_sd, 1),
                   lfp_shifted_uv=round(lfp_sd_shift, 1)))
CK = pd.DataFrame(ck)
print(CK.to_string(index=False))
print("Median DOWN/UP rate ratio over files: true", CK.down_up_ratio.median(), "shifted 10 s", CK.down_up_ratio_shifted.median())
print("Transition-averaged LFP (largest SD over time, uV): median", CK.lfp_transition_uv.median(),
      "; with onsets shifted by 10 s", CK.lfp_shifted_uv.median())
print("Units tagged by the laser pulses (pooled rule): true onsets", int(U.opto_pooled.sum()),
      "; onsets shifted by 0.5 s", int(U.opto_pooled_null.sum()))
wk = np.array([np.nanmedian(r["state_rates"]["Awake"]) for r in S if r["state_rates"] is not None])
nr = np.array([np.nanmedian(r["state_rates"]["Non-REM"]) for r in S if r["state_rates"] is not None])
print("Median unit rate per file, Awake against Non-REM (Hz): ", np.round(wk, 2), np.round(nr, 2))

The checks above are what the data can show about their clocks. Spikes fall far less in the dataset's DOWN intervals than in its UP intervals, and the contrast goes when the intervals are shifted by 10 s, so the UP/DOWN table shares the spike clock. Every UP or DOWN interval lies inside a Non-REM interval of the state table, which is consistent with the state table sharing that clock (in YMV19, which has no state table, this cannot be checked). The LFP averaged around DOWN-UP onsets swings far more than around the same onsets shifted by 10 s, so the LFP shares the clock of the UP/DOWN table and, through it, of the spikes. Units respond to the laser pulses and not to the pulses shifted by 0.5 s, so the pulse table shares the spike clock.

## fig1g-landmark-distances-consistent (blocked, methods_underspecified; every reading shown)

> The distances between the five depth landmarks (the MUA power peak c and the CSD landmarks a, b, d, e) vary by less than 100 um from mouse to mouse.

Criterion: c is the depth of the peak of 500 Hz to 5 kHz power; a, b, d, e are the DOWN-UP CSD landmarks as in fig1f-sink-b-deeper (the most negative CSD near the layer 3/4 border within 0 to 100 ms, reading a, or 0 to 50 ms, reading b, after the onset; sources as the most positive CSD near the stated depth); "varied" is read as the range across mice of each distance between adjacent landmarks, or each mouse's deviation from the across-mouse mean. Reproduced when, under a reading, every adjacent-landmark distance varies by less than 110 um.

Here the DOWN-UP transitions are the dataset's UP onsets that follow a DOWN interval, averaged over the 1000 s of each session that hold the most of them; MUA power is the 500 Hz to 5 kHz power of the raw signal in the two 30 s waveform blocks. The paper does not give a rule that places a, b, d and e (Results, Anchoring Mesoscopic Patterns, paragraph 1: "the most prominent current source density (CSD) sinks and sources"; Figure 1D marks them by hand with asterisks). The rule used is: b the most negative CSD above c and d below it, a the most positive above b and e below d. The claims file applies no spatial smoothing (main); a second difference over two sites (40 um) is shown as a variant, as single-site deviations dominate the unsmoothed CSD.

In [ ]:
lm_rows = []
for r in S:
    L = r["landmark"]
    for step in (1, 2):
        cs = v.csd_of(L["lfp"], step)
        for w in v.CSD_WINDOWS:
            m = v.landmarks(cs, L["lags"], r["mua_power"], w, step)
            row = dict(subject=r["subject"], transitions=L["n"], step=step, window=f"{w[0]*1000:.0f}-{w[1]*1000:.0f} ms",
                       c=int(np.argmax(r["mua_power"])))
            if m:
                row.update({k: m[k] for k in "abde"})
                row.update({"a-b": (m["b"] - m["a"]) * v.SITE_UM, "b-c": (m["c"] - m["b"]) * v.SITE_UM,
                            "c-d": (m["d"] - m["c"]) * v.SITE_UM, "d-e": (m["e"] - m["d"]) * v.SITE_UM})
            lm_rows.append(row)
LM = pd.DataFrame(lm_rows)
print("MUA peak site (c) by mouse:", dict(zip(LM.subject[LM.step == 1][::2], LM.c[LM.step == 1][::2])))
print("Transitions averaged per mouse: median", int(LM.transitions.median()), "range", LM.transitions.min(), "to", LM.transitions.max())
for (step, w), g in LM.groupby(["step", "window"]):
    ok = g.dropna(subset=["a-b"])
    d = ok[["a-b", "b-c", "c-d", "d-e"]]
    rng = (d.max() - d.min())
    dev = (d - d.mean()).abs().max()
    print(f"\nCSD over +-{step} site(s), window {w}: landmarks placed in {len(ok)} of {len(g)} mice "
          f"(no site above or below c in {len(g) - len(ok)})")
    print("  distance medians (um):", d.median().round(0).to_dict())
    print("  range across mice (um):", rng.round(0).to_dict(), "; all < 110:", bool((rng < 110).all()))
    print("  largest deviation from the mean (um):", dev.round(0).to_dict(), "; all < 110:", bool((dev < 110).all()))
print()
print(LM[(LM.step == 1) & (LM.window == "0-100 ms")].drop(columns=["step", "window"]).to_string(index=False))
r0 = S[4]
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
cs = v.csd_of(r0["landmark"]["lfp"], 1)
lim = np.percentile(np.abs(cs), 99)
ax[0].imshow(cs.T, aspect="auto", cmap="RdBu", vmin=-lim, vmax=lim,
             extent=[r0["landmark"]["lags"][0] * 1000, r0["landmark"]["lags"][-1] * 1000, 62.5, 0.5])
ax[0].set_xlabel("ms from DOWN-UP"); ax[0].set_ylabel("site"); ax[0].set_title(r0["subject"] + " CSD (sinks red)")
ax[1].plot(r0["mua_power"] / r0["mua_power"].max(), np.arange(64)); ax[1].invert_yaxis()
ax[1].set_xlabel("MUA power (norm.)"); plt.tight_layout(); plt.show()
print(f"CSD colour scale clipped at the 99th percentile of |CSD| ({lim:.1f} uV per site^2); {int((np.abs(cs) > lim).sum())} of {cs.size} values beyond it")

**Result: blocked (methods_underspecified).** Under every reading the landmarks cannot be placed in 3 or 4 of the 19 mice, and among those placed the adjacent-landmark distances range across mice by 220 to 620 um, far beyond 110 um. In several mice landmark a falls one site above b, the mark of a single-site deviation in the unsmoothed CSD, and the two-site CSD does not narrow the ranges. The paper chose the landmarks from the CSD maps by eye ("the most prominent ... sinks and sources", Figure 1D), so the rule run here is not the paper's procedure, and no rule given in the paper or the claims file places them; the result of every reading is shown above.

## figs7-strongest-3-6hz-coupling

> The strongest spike-LFP coupling is in the 3-6 Hz band, followed by the slow gamma (30-60 Hz) band.

Criterion: per unit in waking, the frequency of largest mean resultant length (preferred frequency, STAR Methods) over log-spaced frequencies from 0.5 to 100 Hz; "strongest" read as (a) the histogram of preferred frequencies of significantly modulated units or (b) the mean modulation strength over units at each frequency; reproduced when, under a reading, the largest value falls in 3-6 Hz and the next in 30-60 Hz, among bands 0.5-3, 3-6, 6-15, 15-30, 30-60 and 60-100 Hz.

Method details taken from the STAR Methods (Spectral analysis, cross-frequency coupling, and spike-LFP coupling): complex Morlet wavelets (Torrence and Compo 1998; w0 = 6 is their standard), phase at each spike, mean resultant length, Rayleigh p < 0.05, preferred frequency as the largest mean vector length. Neither the Methods nor the Figure S7 legend say which LFP site the phases come from. Two readings are run: the unit's own peak site, and the site 10 sites (200 um) away from it, which keeps the unit's own spike out of the LFP (the 1250 Hz LFP of the unit's site still holds part of its spike). The second is adopted as an assumption. Waking is the `Awake` intervals of at least 30 s, up to 600 s per session (the longest), with spikes within 6 s of an epoch edge left out; units with fewer than 50 spikes there are left out. 25 log-spaced frequencies; the LFP is decimated to 312.5 Hz. YMV19 has no state table and is left out. A unit is significantly modulated when the Rayleigh p at its preferred frequency is below 0.05.

In [ ]:
bands = [(0.5, 3), (3, 6), (6, 15), (15, 30), (30, 60), (60, 100.01)]
bnames = ["0.5-3", "3-6", "6-15", "15-30", "30-60", "60-100"]
F = v.COUPLING_FREQS
print("Frequencies (Hz):", np.round(F, 2))
cp = [(si, r["coupling"]) for si, r in enumerate(S) if r["coupling"] is not None]
print("Sessions with waking coupling:", len(cp), "; waking seconds used: median", np.median([c["seconds"] for _, c in cp]),
      "range", min(c["seconds"] for _, c in cp), "to", max(c["seconds"] for _, c in cp))
for reading in ("own", "distant"):
    pref, mrl_all, subj, n_drop, n_units = [], [], [], 0, 0
    for si, c in cp:
        ok = c["n_spikes"] >= 50
        n_drop += int((~ok).sum()); n_units += len(ok)
        m = np.nan_to_num(c[f"mrl_{reading}"], nan=-1)[ok]
        p = c[f"p_{reading}"][ok]
        i = np.argmax(m, axis=1)
        sig = p[np.arange(len(i)), i] < 0.05
        pref += list(F[i][sig]); subj += [S[si]["subject"]] * int(sig.sum())
        mrl_all.append(m)
    pref = np.array(pref); mrl_all = np.vstack(mrl_all)
    hist = [int(((pref >= a) & (pref < b)).sum()) for a, b in bands]
    mean = [mrl_all[:, (F >= a) & (F < b)].mean() for a, b in bands]
    print(f"\nLFP of the {reading} site: {len(mrl_all)} units ({n_drop} of {n_units} left out, < 50 waking spikes), "
          f"{len(pref)} significantly modulated")
    print("  (a) preferred frequencies by band:", dict(zip(bnames, hist)),
          "; order:", [bnames[j] for j in np.argsort(hist)[::-1]][:3])
    print("  (b) mean MRL by band:", dict(zip(bnames, np.round(mean, 4))),
          "; order:", [bnames[j] for j in np.argsort(mean)[::-1]][:3])
    if reading == "distant":
        P = pd.DataFrame({"subject": subj, "band": pd.cut(pref, [a for a, _ in bands] + [100.01], labels=bnames, right=False)})
        print("  preferred band by mouse (counts):")
        print(pd.crosstab(P.subject, P.band).to_string())
        print(f"  fraction significant at the preferred frequency: {len(pref) / len(mrl_all):.2f}; with 25 frequencies "
              f"tested, chance would give at most {1 - 0.95 ** 25:.2f} (independent frequencies)")
fig, ax = plt.subplots(figsize=(5, 3))
for reading, col in (("own", "0.6"), ("distant", "k")):
    mm = np.vstack([np.nan_to_num(c[f"mrl_{reading}"], nan=np.nan)[c["n_spikes"] >= 50] for _, c in cp])
    ax.semilogx(F, np.nanmean(mm, axis=0), color=col, label=reading)
ax.set_xlabel("Hz"); ax.set_ylabel("mean MRL"); ax.legend(); plt.tight_layout(); plt.show()

**Result: partially reproduced, resting on an assumption about the LFP site.** With the LFP 10 sites away from each unit (the adopted reading), the mean modulation strength is largest in 3-6 Hz (0.072), as the claim says, but the next band is 0.5-3 Hz (0.063), not 30-60 Hz (0.048, fifth of six). By the histogram of preferred frequencies, 0.5-3 Hz comes first (590 units), 3-6 Hz second (346) and 30-60 Hz fifth (71). With the LFP of the unit's own site, 60-100 Hz comes first under both readings (716 units; mean 0.107), which fits the unit's own spike leaking into the LFP. Under no reading does 30-60 Hz come second.

## fig4a-unit-counts

> Clustering one session from each of the 19 mice yields 1,472 units, with 251 narrow-waveform putative interneurons and 1,075 wide-waveform putative principal cells.

Criterion: units as sorted in the dataset; classes by the waveform rules of the STAR Methods, trough-to-peak latency from the mean waveform on the peak channel. The dataset holds no waveforms, so the mean waveform of each unit is averaged from 60 s of the raw signal (two 30 s blocks, at 1000 s and 3000 s), on all 64 sites; the peak site is the one with the largest peak-to-peak amplitude. The Results say "the unfiltered waveform was quantified by the trough-to-peak latency" (Assignment of Neurons to Physiologically Defined Layers, paragraph 1), so the main reading is the wide-band signal; the 300 Hz high-pass that Kilosort used (STAR Methods, Spike sorting, `ops.fshigh 300`) is a variant. Units with fewer than 10 spikes in the blocks are left unclassified.

In [ ]:
print("Units in the files:", len(U), "; with quality 'good':", int((U.quality == "good").sum()))
for col, name in (("wf", "wide band (main)"), ("wf_hp", "300 Hz high-pass (variant)")):
    print(f"Waveform classes, {name}:", U[col].value_counts().to_dict())
print("Spikes used per unit for the waveform: median", int(U.wf_spikes.median()), "; units with fewer than 10:",
      int((U.wf_spikes < 10).sum()))
neg = U.wf.isin(["E", "I"])
print("\nThreshold sensitivity (wide band, negative non-fiber units):")
for th in (0.45, 0.50, 0.55, 0.60, 0.65):
    print(f"  TP <= {th:.2f} ms: I {int((U.tp_ms[neg] <= th).sum())}, E {int((U.tp_ms[neg] > th).sum())}")
print("Trough-to-peak latencies are quantized at", 1000 / v.RAW_RATE, "ms (20 kHz); range",
      U.tp_ms.min(), "to", U.tp_ms.max(), "ms")
print("\nBy mouse (wide band):")
print(pd.crosstab(U.subject, U.wf, margins=True).T.to_string())
print("\nAgainst the dataset's own cell_type labels (wide band):")
print(pd.crosstab(U.wf, U.cell_type).to_string())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(U.tp_ms[neg], bins=np.arange(0, 1.6, 0.05), color="0.4"); ax[0].axvline(0.55, color="r")
ax[0].set_xlabel("trough-to-peak (ms), wide band"); ax[0].set_ylabel("units")
ax[1].hist(U.tp_hp[U.wf_hp.isin(["E", "I"])], bins=np.arange(0, 1.6, 0.05), color="0.4"); ax[1].axvline(0.55, color="r")
ax[1].set_xlabel("trough-to-peak (ms), 300 Hz high-pass")
plt.tight_layout(); plt.show()
print("Units outside the 0 to 1.6 ms axes: wide band", int((U.tp_ms[neg] >= 1.6).sum()),
      "; high-pass", int((U.tp_hp[U.wf_hp.isin(['E', 'I'])] >= 1.6).sum()))

**Result: reproduced.** The files hold 1,477 units (1,476 of quality "good"), within 1,472 ± 147. From the wide-band waveform, as the Results describe, 246 units are narrow-waveform I cells (251 ± 25) and 1,016 wide-waveform E cells (1,075 ± 108); 105 are positive units, no unit is a fiber by the kurtosis rule, and 110 have fewer than 10 spikes in the 60 s used. With a 300 Hz high-pass the counts are 320 and 941, both outside tolerance, and the I count also leaves tolerance at thresholds of 0.60 ms (323) or 0.50 ms (215).

## fig4a-ccg-classification

> Short-latency CCG interactions identify 347 E and 100 I cells, and these agree with the waveform classes (97.4% of CCG-based E cells are waveform E cells; 98.0% of CCG-based I cells are waveform I cells).

Criterion: a unit is a CCG-based E (I) cell when it is the presynaptic unit of at least one significant peak (trough) by the monosynaptic rule of the STAR Methods: CCGs in 0.5 ms bins over the whole session, a Gaussian of 7 ms SD as the predictor, the 99.9999th Poisson percentile (0.0001th for troughs), at least two consecutive bins within +1.5 to +4 ms. The unhollowed Gaussian (Methods) is the main reading and the buzcode hollowed one (0.6) a variant (the claims file's opening comment). The percentages are taken over all CCG-based cells (main) and over those with an E or I waveform class (variant). The dataset's own `synaptic_effect` labels are shown for comparison.

In [ ]:
for e, i, name in (("ccgE", "ccgI", "unhollowed (main)"), ("ccgE_h", "ccgI_h", "hollowed 0.6 (variant)")):
    nE, nI = int(U[e].sum()), int(U[i].sum())
    pE = (U.wf[U[e]] == "E").mean() * 100
    pI = (U.wf[U[i]] == "I").mean() * 100
    pE2 = (U.wf[U[e] & U.wf.isin(["E", "I"])] == "E").mean() * 100
    pI2 = (U.wf[U[i] & U.wf.isin(["E", "I"])] == "I").mean() * 100
    print(f"{name}: CCG-based E {nE}, I {nI}, both {int((U[e] & U[i]).sum())}; "
          f"E that are waveform E {pE:.1f}% (of classified {pE2:.1f}%), I that are waveform I {pI:.1f}% (of classified {pI2:.1f}%)")
print("\nCCG-based E cells by waveform class:", U.wf[U.ccgE].value_counts().to_dict())
print("CCG-based I cells by waveform class:", U.wf[U.ccgI].value_counts().to_dict())
print("\nBy mouse (unhollowed):")
print(U.groupby("subject").agg(units=("unit", "size"), ccgE=("ccgE", "sum"), ccgI=("ccgI", "sum"),
                               ds_exc=("synaptic_effect", lambda s: (s == "Excitatory").sum()),
                               ds_inh=("synaptic_effect", lambda s: (s == "Inhibitory").sum())).T.to_string())
print("\nThe dataset's synaptic_effect labels against this detection:")
print(pd.crosstab(U.synaptic_effect, [U.ccgE.rename("ccgE"), U.ccgI.rename("ccgI")]).to_string())
lab = U.synaptic_effect.isin(["Excitatory", "Inhibitory"])
print("Dataset labels: Excitatory", int((U.synaptic_effect == "Excitatory").sum()), "Inhibitory",
      int((U.synaptic_effect == "Inhibitory").sum()), "; Excitatory that are waveform E",
      round((U.wf[U.synaptic_effect == "Excitatory"] == "E").mean() * 100, 1), "%; Inhibitory that are waveform I",
      round((U.wf[U.synaptic_effect == "Inhibitory"] == "I").mean() * 100, 1), "%")

**Result: not reproduced.** The STAR Methods rule gives 673 CCG-based E cells and 187 I cells (347 ± 35 and 100 ± 10), 122 of them both. Of the CCG-based E cells 74.9% are waveform E cells (80.9% of those with a class), and of the I cells 72.7% are waveform I cells (79.5%), against 97.4% and 98.0%. The hollowed predictor changes little (682 and 188). The dataset's own synaptic_effect labels count 444 excitatory and 145 inhibitory units, also more than the paper's.

## fig7-connection-counts

> Of 102,332 potential neuron pairs, 525 putative excitatory and 323 putative inhibitory monosynaptic connections were identified.

Criterion: detection by the STAR Methods rule. The pair count is read as ordered pairs within a session among (a) all units or (b) E and I units only (waveform classes), and as unordered pairs of each. The Gaussian baseline is run unhollowed (Methods) and hollowed 0.6 (buzcode). Reproduced when, under a reading, all three counts are within tolerance (102,332 ± 10,233; 525 ± 53; 323 ± 32).

The site distance between the two units of each connection is shown as a diagnostic: units 0 or 1 site apart (within 20 um) can share spikes after sorting, which would add peaks the paper may not have counted. It is not a criterion.

In [ ]:
n_all = U.groupby("sess").size().values
n_ei = U[U.wf.isin(["E", "I"])].groupby("sess").size().reindex(range(len(S)), fill_value=0).values
pairs = {"ordered, all units": int((n_all * (n_all - 1)).sum()), "ordered, E and I units": int((n_ei * (n_ei - 1)).sum()),
         "unordered, all units": int((n_all * (n_all - 1) // 2).sum()), "unordered, E and I units": int((n_ei * (n_ei - 1) // 2).sum())}
for k, val in pairs.items():
    print(f"potential pairs, {k}: {val}  (within 102,332 +/- 10,233: {abs(val - 102332) <= 10233})")
for key, name in (("conn", "unhollowed (main)"), ("conn_hollow", "hollowed 0.6")):
    ne = int(sum(r[key]["exc"].sum() for r in S)); ni = int(sum(r[key]["inh"].sum() for r in S))
    print(f"{name}: excitatory {ne} (525 +/- 53: {abs(ne - 525) <= 53}), inhibitory {ni} (323 +/- 32: {abs(ni - 323) <= 32})")
# among E and I units only
ei = C[C.pre_wf.isin(["E", "I"]) & C.post_wf.isin(["E", "I"])]
print("Connections between units with an E or I waveform class: excitatory", int(ei.exc.sum()), "inhibitory", int(ei.inh.sum()))
print("\nBy mouse:")
print(C.groupby("subject").agg(exc=("exc", "sum"), inh=("inh", "sum")).T.to_string())
print("\nSite distance between pre and post units (sites of 20 um):")
print(pd.DataFrame({"excitatory": C[C.exc].dsite.clip(upper=10).value_counts().sort_index(),
                    "inhibitory": C[C.inh].dsite.clip(upper=10).value_counts().sort_index()}).T.to_string())
far = C[C.dsite >= 2]
print("Connections with units at least 2 sites apart: excitatory", int(far.exc.sum()), "inhibitory", int(far.inh.sum()))

**Result: not reproduced.** The STAR Methods rule finds 1,295 excitatory and 578 inhibitory connections (525 ± 53 and 323 ± 32), and 1,328 and 591 with the hollowed predictor. No reading of the pair count is within 102,332 ± 10,233: ordered pairs of all units give 121,842, of E and I units 89,046, and unordered pairs 60,921 and 44,523. Of the excitatory connections 482 join units 0 or 1 site apart; with units at least 2 sites apart there remain 813 and 462, still above the paper's counts.

## fig7c-divergence-convergence

> Most presynaptic principal neurons have a single target in the recorded population, a few target several; an interneuron typically has one presynaptic partner, a few many, up to 16.

Criterion: from the E-I connections (excitatory connections from a waveform E unit to a waveform I unit), the count of targets per presynaptic E cell and of presynaptic E cells per I cell. Reproduced when the mode of both distributions is one and the largest convergence is within 16 ± 1.6.

In [ ]:
EI = C[C.exc & (C.pre_wf == "E") & (C.post_wf == "I")]
div = EI.groupby(["sess", "pre"]).size()
con = EI.groupby(["sess", "post"]).size()
print("E-I connections:", len(EI), "from", len(div), "E cells to", len(con), "I cells")
print("targets per presynaptic E cell:", div.value_counts().sort_index().to_dict(), "; mode", int(div.mode()[0]))
print("presynaptic E cells per I cell:", con.value_counts().sort_index().to_dict(), "; mode", int(con.mode()[0]),
      "; largest", int(con.max()))
print("I cells with more than 16 presynaptic E cells:", int((con > 16).sum()))
print("Largest convergence by mouse:", con.groupby(level=0).max().rename(lambda i: S[i]["subject"]).to_dict())
EIh = []
for si, r in enumerate(S):
    u = U[U.sess == si].wf.values
    ex = r["conn_hollow"]["exc"] & (u == "E")[:, None] & (u == "I")[None, :]
    EIh.append(ex.sum(axis=0)[ex.sum(axis=0) > 0])
conh = np.concatenate(EIh)
print("Hollowed variant: presynaptic E cells per I cell, mode", int(stats.mode(conh).mode), "; largest", int(conh.max()))

**Result: partially reproduced.** Of 549 E-I connections, most presynaptic E cells have one target (242 of 362, mode 1), and the most common number of presynaptic E cells per I cell is one (39 of 138 I cells, mode 1). The largest convergence is 29 (one I cell in YMV07; 18 for the next), outside 16 ± 1.6, so the claim holds in its shape but not in its maximum.

## figs8f-ie-dip-wake

> Over all I-E pairs, spike suppression is stronger in waking than in non-REM sleep (dip 24.1% against 22.8% of the CCG baseline).

Criterion: for each I-E connection and state, the dip as (predictor minus CCG) over predictor at the deepest bin in +1.5 to +4 ms; means and medians over pairs are both readings; reproduced when both values are within tolerance (24.1 ± 2.41 and 22.8 ± 2.28%) and waking > non-REM by a two-sided signed-rank test at p < 0.05. The CCG of a state uses the presynaptic spikes in that state (`Awake` or `Non-REM` intervals) and its own predictor. Which connections are "I-E" is taken from the Results (Within-Layer and Inter-layer Interactions among Neurons, paragraph 1), where the I-E connections number 323, the count of all inhibitory connections: the main reading is every inhibitory connection, and the variant those from a waveform I unit to a waveform E unit. YMV19 has no state table and is left out.

In [ ]:
for name, sel in (("all inhibitory connections (main)", C.inh),
                  ("waveform I to waveform E (variant)", C.inh & (C.pre_wf == "I") & (C.post_wf == "E"))):
    x = C[sel]
    ok = x.dip_wake.notna() & x.dip_nrem.notna()
    x = x[ok]
    w = stats.wilcoxon(x.dip_wake, x.dip_nrem)
    print(f"{name}: {len(x)} pairs ({int((~ok).sum())} without spikes in a state); dip wake mean {x.dip_wake.mean()*100:.1f}%, "
          f"non-REM mean {x.dip_nrem.mean()*100:.1f}%; medians {x.dip_wake.median()*100:.1f}% and {x.dip_nrem.median()*100:.1f}%; "
          f"wake > non-REM in {int((x.dip_wake > x.dip_nrem).sum())} pairs; signed-rank p = {w.pvalue:.3g}; "
          f"median difference {np.median(x.dip_wake - x.dip_nrem)*100:.2f} points")
x = C[C.inh & C.dip_wake.notna() & C.dip_nrem.notna()]
print("\nBy mouse (main reading):")
print(x.groupby("subject").agg(pairs=("pre", "size"), wake=("dip_wake", "median"), nrem=("dip_nrem", "median")).round(3).T.to_string())
print("Dip range (wake, non-REM):", round(x.dip_wake.min(), 3), round(x.dip_wake.max(), 3), round(x.dip_nrem.min(), 3), round(x.dip_nrem.max(), 3))

**Result: not reproduced.** Over 563 inhibitory connections with spikes in both states, the dip is 39.6% of the predictor in waking and 39.9% in non-REM (medians 35.6% and 36.0%), outside 24.1 ± 2.41% and 22.8 ± 2.28% under either reading, and waking does not exceed non-REM (signed-rank p = 0.37; 265 of 563 pairs larger in waking). The strict I-to-E reading gives 40.0% and 39.4% (p = 0.57).

## fig4de-optotagging

> Fourteen of 15 optogenetically identified PV cells and 40 of 52 VGAT cells fall in the narrow-waveform inhibitory group.

Criterion: tagging by the STAR Methods rule (1 ms PSTH bins, mean of the two largest bins in 1 to 6 ms after the 10 ms pulse above the baseline mean plus 8 SD of -100 to 0 ms) in the PV-Cre::Ai32 and VGAT-Cre::Ai32 mice; readings, all pulses pooled, and each intensity separately with a unit tagged when any passes. The pulses are the dataset's `Laser diode` intervals (10 ms; three amplitude levels, read as the three intensities). The mice are taken by the genotype in the files, which gives 11 PV-Cre::Ai32 and 3 VGAT-Cre::Ai32 mice where the paper has 5 and 9. Units tagged in the other genotypes and with onsets shifted by 0.5 s are shown as references.

In [ ]:
for col, name in (("opto_pooled", "pooled (main)"), ("opto_any", "any intensity")):
    for g in ("PV-Cre::Ai32", "VGAT-Cre::Ai32"):
        x = U[(U.genotype == g) & U[col]]
        print(f"{name}, {g}: tagged {len(x)}, in the waveform I group {int((x.wf == 'I').sum())}, "
              f"waveform E {int((x.wf == 'E').sum())}, other {int((~x.wf.isin(['E', 'I'])).sum())}")
    oth = U[~U.genotype.isin(["PV-Cre::Ai32", "VGAT-Cre::Ai32"]) & U[col]]
    print(f"{name}, other genotypes: tagged {len(oth)}; shifted onsets, all mice: {int(U[col + '_null'].sum())}")
print("\nBy mouse (pooled):")
print(U[U.sess.map(lambda i: S[i]["opto"] is not None)].groupby(["subject", "genotype"])
      .agg(units=("unit", "size"), tagged=("opto_pooled", "sum"),
           tagged_I=("opto_pooled", lambda s: int((s & (U.loc[s.index, "wf"] == "I")).sum()))).to_string())
both = U.genotype.isin(["PV-Cre::Ai32", "VGAT-Cre::Ai32"])
print("\nPV and VGAT mice together: tagged", int(U[both].opto_pooled.sum()), "in I group",
      int((U[both].opto_pooled & (U[both].wf == "I")).sum()), "(paper 67 and 54)")

**Result: not reproduced.** In the 11 mice the files label PV-Cre::Ai32, 62 units pass the tagging rule with all pulses pooled, 48 of them in the waveform I group (paper 15 and 14). In the 3 VGAT-Cre::Ai32 mice 18 pass, 14 in the I group (paper 52 and 40). Testing each intensity gives 61 and 18. No unit passes in the other genotypes or with onsets shifted by 0.5 s. Taken together, 80 units are tagged and 62 of them (78%) are in the I group, against the paper's 67 and 54 (81%). The split by genotype differs from the paper's, as do the genotype counts in the files (11 PV and 3 VGAT mice against 5 and 9).

## figs8c-lognormal-strength (blocked, methods_underspecified)

> Connection strengths are skewed and lognormally distributed, a few strong connections mixed with many weak ones.

The criterion records this claim as methods_underspecified: the paper reports no test or measure of lognormality. The distribution of log transmission probabilities is shown with its skewness, as description.

In [ ]:
for name, sel in (("excitatory", C.exc), ("inhibitory", C.inh)):
    s_ = C[sel].stp.abs()
    s_ = s_[s_ > 0]
    print(f"{name}: {len(s_)} connections; |transmission probability| median {s_.median():.4f}, range {s_.min():.5f} to {s_.max():.3f}; "
          f"skewness of raw values {stats.skew(s_):.2f}, of log10 values {stats.skew(np.log10(s_)):.2f}")
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(np.log10(C[C.exc].stp[C[C.exc].stp > 0]), bins=30, color="0.4")
ax.set_xlabel("log10 transmission probability, excitatory"); ax.set_ylabel("connections"); plt.tight_layout(); plt.show()
print("Excitatory connections with transmission probability <= 0 (not plotted):", int((C[C.exc].stp <= 0).sum()))

## Blocked claims: what was tried

Most claims of the paper rest on depth landmarks and laminar layers computed from the LFP and the raw signal: the CSD landmarks of DOWN-UP transitions, the MUA power peak, gamma coherence clustering, gamma ICA, and the K-means clustering of unit-IC coupling that defines the eight layer groups. None of these is in the dataset (no layer, depth or cluster label per unit or site; the electrode table has no depths), so each would have to be computed from the LFP of all 19 sessions, starting from the landmarks, which fig1g above could not place by a rule. The cell below times a read of the LFP on one file and estimates the read of the whole LFP, to which the analyses would add; it also shows, for one file, how often each site's most correlated neighbour in the 100-500 Hz LFP is an adjacent site, a check of the site order that the electrode table does not give.

In [ ]:
f = v.open_asset(assets[0]["asset_id"])
L = f["processing/ecephys/LFP/LFP/data"]
t0 = time.time()
x = L[0:143802 * 2, :].astype(float)
dt = time.time() - t0
sec = x.shape[0] / 1250
total_s = sum(r["lfp_shape"][0] / r["lfp_rate"] for r in S)
print(f"read {sec:.0f} s of 64-site LFP in {dt:.1f} s; whole LFP of the 19 files: {total_s/3600:.1f} h of recording,"
      f" estimated read {total_s / sec * dt / 60:.0f} min at this rate")
from scipy.signal import butter, sosfiltfilt
xh = sosfiltfilt(butter(3, [100, 500], btype="band", fs=1250, output="sos"), x, axis=0)
cc = np.corrcoef(xh.T); np.fill_diagonal(cc, -1)
best = np.argmax(cc, axis=1) - np.arange(64)
print("most correlated site (100-500 Hz LFP) is an adjacent one for", int((np.abs(best) == 1).sum()), "of 64 sites in", S[0]["subject"])
f.close()

## Summary

| Claim | Outcome |
|---|---|
| fig4a-unit-counts | reproduced |
| fig4a-ccg-classification | not reproduced |
| fig7-connection-counts | not reproduced |
| fig7c-divergence-convergence | partially reproduced |
| figs8f-ie-dip-wake | not reproduced |
| fig4de-optotagging | not reproduced |
| figs7-strongest-3-6hz-coupling | partially reproduced (assumed LFP site) |
| fig1g-landmark-distances-consistent | blocked, methods underspecified |
| figs8c-lognormal-strength, fig3ab-alpha-analogy | blocked, methods underspecified (criterion) |
| fig1f-sink-b-deeper, fig1f-other-landmarks-same-depth, figs2d-visual-offset-rebound | blocked, data missing |
| 4 claims whose LFP site is placed by the landmarks | blocked, methods underspecified |
| 30 claims that depend, directly or through other claims, on fig1g-landmark-distances-consistent | blocked, methods underspecified (inherited) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: mean waveforms come from 60 s of raw signal per session, not from all spikes; the monosynaptic rule is the STAR Methods one with a 50 ms CCG half width, and no further selection of pairs; brain states are the dataset's `Sleep states` intervals with `Awake` as waking; tagging uses the dataset's `Laser diode` pulses and the genotypes written in the files; spike-LFP coupling uses up to 600 s of waking per session, on an LFP site the paper does not name; the landmarks are placed by a rule, where the paper placed them by eye.

What was found about the dataset: the visual stimulus times are in 2 of the 13 mice the paper stimulated, and the `Visual laser` tables of those two also hold the optogenetic pulses; the genotypes in the files (11 PV-Cre::Ai32, 3 VGAT-Cre::Ai32) disagree with the paper (5 and 9); the electrode table gives no depth (y is NaN, location "unknown"); the units carry no waveforms and no layer or cluster label; `max_channel` matches the site of the largest mean waveform for 76 of 1,367 units, so its indexing is undocumented; the `cell_type` labels (pyramidal, narrow and wide interneuron, "down state") follow a scheme the paper does not describe, and 10 units in 6 mice are labelled "down state" where the paper reports 5 neurons in 4 mice; one file (YMV03) lacks every unit column but quality and one (YMV19) lacks the state table. No recording appears twice, and the two published versions hold the same files.